[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Server-Side Cursors &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the server and the `grew_by` helper. Run it first.
Task 3 takes a few seconds, because it runs two measurements in two separate processes.


In [1]:
import getpass
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

MEASURING = """
import resource, sys, psycopg

def resident():
    raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss     # bytes on macOS, kilobytes on Linux
    return (raw if sys.platform == "darwin" else raw * 1024) / 1_000_000

QUERY = "SELECT n, repeat('x', 20) FROM generate_series(1, ROWS) AS n"
before = resident()

with psycopg.connect("dbname=guide") as conn:
    if "KIND" == "fetchall":
        with conn.cursor() as cur:
            cur.execute(QUERY)
            cur.fetchall()
    elif "KIND" == "fetchmany":
        with conn.cursor() as cur:
            cur.execute(QUERY)
            while cur.fetchmany(1000):
                pass
    elif "KIND" == "server":
        with conn.cursor(name="batch") as cur:
            cur.itersize = 1000
            cur.execute(QUERY)
            for _ in cur:
                pass
    elif "KIND" == "stream":
        with conn.cursor() as cur:
            for _ in cur.stream(QUERY):
                pass

print(round((resident() - before) / 25) * 25)                    # coarse, so two runs agree
"""


def grew_by(kind, rows=300_000):
    """How much the process grew reading the rows this way, in megabytes.

    Each way is measured in a Python of its own, because resident memory is a high-water mark that
    never falls, so measuring two ways in one process would report the larger of them twice. The
    answer is rounded to the nearest twenty five, because the exact number depends on the machine
    and on the allocator, and what this notebook is about is the difference between the four.
    """
    script = MEASURING.replace("KIND", kind).replace("ROWS", str(rows))
    done = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
    return int(done.stdout.strip())


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


**1.** What each cursor knows before you fetch.


In [2]:
query = "SELECT n FROM generate_series(1, 50000) AS n"

with psycopg.connect("dbname=guide") as conn:
    with conn.cursor() as ordinary:
        ordinary.execute(query)
        print("ordinary cursor rowcount:", ordinary.rowcount)

    with conn.cursor(name="held") as named:
        named.execute(query)
        print("named cursor rowcount:   ", named.rowcount)


ordinary cursor rowcount: 50000
named cursor rowcount:    -1


The ordinary cursor can count because the rows are already in this process. The named one answers
`-1`, which is psycopg saying it does not know: the server has been told to declare a cursor and has
not been asked for a row yet.


**2.** Which of them the server knows about.


In [3]:
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor() as cur:
        cur.execute(query)
        print("with an ordinary cursor, pg_cursors holds:",
              conn.execute("SELECT count(*) FROM pg_cursors").fetchone()[0])

    with conn.cursor(name="mine") as cur:
        cur.execute(query)
        names = [row[0] for row in conn.execute("SELECT name FROM pg_cursors")]
        print("with a named one, pg_cursors holds:     ", names)


with an ordinary cursor, pg_cursors holds: 0
with a named one, pg_cursors holds:      ['mine']


`pg_cursors` lists what the server is holding open. An ordinary cursor never appears, because by the
time `execute` returned the server had finished with the statement entirely.


**3.** What two of the ways cost.


In [4]:
for kind, description in (("fetchall", "fetchall()"), ("server", "a named cursor")):
    print(f"  {description:<16} grew the process by about {grew_by(kind, rows=200_000):>3} MB")


  fetchall()       grew the process by about  50 MB
  a named cursor   grew the process by about   0 MB


Same rows, same query, and the difference is where they were held. The named cursor's number is
close to nothing because only one batch was ever in this process at a time.


**4.** Batches from a named cursor.


In [5]:
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor(name="threes") as cur:
        cur.execute("SELECT n FROM generate_series(1, 10) AS n")
        number = 0
        while batch := cur.fetchmany(3):
            number += 1
            print(f"  batch {number}: {[row[0] for row in batch]}")


  batch 1: [1, 2, 3]
  batch 2: [4, 5, 6]
  batch 3: [7, 8, 9]
  batch 4: [10]


Four batches for ten rows, the last one short. `fetchmany` on a named cursor is a real `FETCH` to the
server, which is what makes the batch size meaningful rather than a way of slicing a list you
already have.


**5.** A stream, read to the end.


In [6]:
with psycopg.connect("dbname=guide") as conn:
    with conn.cursor() as cur:
        seen = 0
        for _ in cur.stream("SELECT n FROM generate_series(1, 20000) AS n"):
            seen += 1

    print("rows read:", seen)
    print("cursors on the server:", conn.execute("SELECT count(*) FROM pg_cursors").fetchone()[0])
    print("transaction:", conn.info.transaction_status.name)


rows read: 20000
cursors on the server: 0
transaction: INTRANS


Nothing was declared and nothing was left behind, because the loop ran to the end. Stopping it early
would have left the connection in `INERROR`, which is why a named cursor is the right tool whenever
stopping early is a possibility.


**6.** An asyncpg cursor, inside what it needs.


In [7]:
conn = await asyncpg.connect(database="guide")

async with conn.transaction():
    rows = [row[0] async for row in conn.cursor("SELECT n FROM generate_series(1, 5) AS n")]
print("inside a transaction:", rows)

try:
    async for _ in conn.cursor("SELECT n FROM generate_series(1, 5) AS n"):
        pass
except asyncpg.exceptions.NoActiveSQLTransactionError as error:
    print("outside one:         ", error)
await conn.close()


inside a transaction: [1, 2, 3, 4, 5]
outside one:          cursor cannot be created outside of a transaction


The same rule psycopg enforces, worded differently. A cursor is something the server holds, and a
transaction is what it is held inside, so asyncpg refuses to make one without a transaction rather
than opening a hidden one.


---

&#8592; **Back to:** [Server-Side Cursors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/07-server-side-cursors.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
